# 01b Pyomo

[Pyomo](https://pyomo.org) reaches PIPS-IPM++ through the `pipsipmpp` solver in
`pyomo.contrib.solver`. Pyomo has no notion of blocks of this kind, so you state the block of
every variable yourself, with a `Suffix`. That makes this the most explicit of the framework
notebooks: you see, and set, every block number.

The PIPS-IPM++ backend is not upstream yet, so the workshop installs Pyomo from a fork.

| sections | run on |
|---|---|
| 1 to 4: build, export, visualize, change the model, reference solve with HiGHS | every machine |
| 5: solve with PIPS-IPM++ | Linux or WSL with the `solver` extra |
| 6: more than one process with MPI | Linux or WSL with the `solver` extra |

In [ ]:
import workshop

workshop.check()

## 1. The model

The sample model from [00](00-start-here.ipynb). The data (costs, profiles, the emission budget)
lives in `sample_data.py` next to this notebook, so every framework computes the same numbers.

The cell below writes the model into a file, `pyomo_model.py`, instead of running it. That way
this notebook and the parallel run in section 6 use the very same code. The cell after it loads
the file.

The annotation is the `pips_block` Suffix at the end of `build`: `0` for the root, `1..N` for a
leaf. Here the hours are cut into `n_blocks` contiguous slices.

In [ ]:
%%writefile pyomo_model.py
import pyomo.environ as pyo

from sample_data import STORAGE, TECHS, budget, profiles


def build(T=24, n_blocks=4, techs=TECHS, storage=False):
    demand, avail = profiles(T)
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)
    m.G = pyo.Set(initialize=list(techs))
    m.cap = pyo.Var(m.G, bounds=(0, None))
    m.gen = pyo.Var(m.G, m.T, bounds=(0, None))

    def supply(m, t):
        return sum(m.gen[g, t] for g in m.G)

    cost = (sum(techs[g][0] * m.cap[g] for g in m.G)
            + sum(techs[g][1] * m.gen[g, t] for g in m.G for t in m.T))

    if storage:
        m.sto_cap = pyo.Var(bounds=(0, None))
        m.charge = pyo.Var(m.T, bounds=(0, None))
        m.discharge = pyo.Var(m.T, bounds=(0, None))
        m.soc = pyo.Var(m.T, bounds=(0, None))
        # the level carries over from the previous hour; the first hour starts empty
        m.soc_level = pyo.Constraint(m.T, rule=lambda m, t: m.soc[t] == (
            m.soc[t - 1] if t > 0 else 0)
            + STORAGE["efficiency"] * m.charge[t] - m.discharge[t])
        m.soc_limit = pyo.Constraint(m.T, rule=lambda m, t: m.soc[t] <= m.sto_cap)
        plain_supply = supply
        supply = lambda m, t: plain_supply(m, t) + m.discharge[t] - m.charge[t]
        cost += STORAGE["investment"] * m.sto_cap

    m.balance = pyo.Constraint(m.T, rule=lambda m, t: supply(m, t) == demand[t])
    m.limit = pyo.Constraint(
        m.G, m.T, rule=lambda m, g, t: m.gen[g, t] <= avail[techs[g][3]][t] * m.cap[g])
    m.carbon = pyo.Constraint(                                            # linking
        expr=sum(techs[g][2] * m.gen[g, t] for g in m.G for t in m.T) <= budget(T))
    m.obj = pyo.Objective(expr=cost, sense=pyo.minimize)

    # the annotation: 0 = root, 1..N = leaf. Every variable needs one.
    m.pips_block = pyo.Suffix(direction=pyo.Suffix.LOCAL)
    hours = -(-T // n_blocks)                                     # hours per block, rounded up
    for g in m.G:
        m.pips_block[m.cap[g]] = 0
        for t in m.T:
            m.pips_block[m.gen[g, t]] = 1 + t // hours
    if storage:
        m.pips_block[m.sto_cap] = 0
        for t in m.T:
            for v in (m.charge, m.discharge, m.soc):
                m.pips_block[v[t]] = 1 + t // hours
    return m

In [ ]:
%run pyomo_model.py
from collections import Counter

model = build()
print("variables per block:", dict(sorted(Counter(model.pips_block.values()).items())))
print("cap[wind] ->", model.pips_block[model.cap["wind"]],
      "| gen[wind,0] ->", model.pips_block[model.gen["wind", 0]],
      "| gen[wind,23] ->", model.pips_block[model.gen["wind", 23]])

Constraints are never annotated. Each one is classified from the variables it uses, which is why
`carbon` becomes a linking row without you saying so.

<details><summary>💡 Hint: I changed the model. Why did nothing happen?</summary>

`%%writefile` saves the code and `%run` loads it. After editing the first cell, run both again,
then build the model anew with `build()`.
</details>

<details><summary>💡 Hint: what happens to a variable I forget to put in the Suffix?</summary>

The backend refuses the model and names the variable, for example
`Variable 'cap[wind]' has no PIPS-IPM++ block annotation`. That is deliberate: guessing a block
could silently give a poor structure. If unsure, put a variable into the root, `0`. A variable
wrongly in the root only widens the border.
</details>

## 2. Export it and look at the matrix

`write_parquet` writes the model the way PIPS-IPM++ reads it: the matrix, the bounds and costs, and
the block of every row and column. It does not need the solver or MPI, so it runs on every
machine.

- `layout="monolithic"` writes one set of files for the whole model. (`"distributed"` writes one
  set per block, see [04](04-deep-dive.ipynb).)
- `names=True` keeps Pyomo's names for every row and column.
- The number of leaves is taken from the largest block number in the Suffix.

In [ ]:
from pathlib import Path
from pyomo.contrib.solver.solvers.pipsipmpp.direct import PipsIpmpp

Path("data").mkdir(exist_ok=True)


def export(model, name, **annotation):
    return PipsIpmpp().write_parquet(model, f"data/{name}", layout="monolithic", names=True,
                                     **annotation)


stem = export(build(), "pyomo_base")
print(sorted(p.name for p in Path("data").glob("pyomo_base.*")))

`workshop.visualize` runs `pipstools visualize` on the files, shows the picture, and prints the
numbers that decide whether a structure is good:

In [ ]:
workshop.visualize(stem)

How to read it:

- Rows and columns are sorted by block. Each dot is a nonzero.
- The narrow band on the left holds the 2 root columns, `cap[wind]` and `cap[gas]`. Its dots are
  the capacity limits of every hour, which use the capacity of their technology.
- The four coloured squares are the leaf blocks, six hours each: 12 `gen` columns and 18 rows
  (6 balance, 12 limit).
- The grey row at the bottom is the one linking row, the carbon budget. It touches every leaf.

This is the structure from the table in 00, now drawn from the actual matrix.

<details><summary>💡 Hint: can I run pipstools myself, in a terminal?</summary>

Yes. `workshop.visualize` runs `pipstools visualize data/pyomo_base.amat.parquet --outputfile ...`.
Run `pipstools --help` from the activated environment (`uv run pipstools --help` or
`pixi run pipstools --help`).
</details>

## 3. Change the model and watch the matrix

Each exercise changes the model once. Before you run a cell, predict what happens to the three
numbers: root columns, columns per leaf, linking rows. Then compare with the picture.

### Exercise 1: more hours

Two days instead of one, still in 4 blocks, and then in 8.

In [ ]:
workshop.visualize(export(build(T=48), "pyomo_T48"))

In [ ]:
workshop.visualize(export(build(T=48, n_blocks=8), "pyomo_T48_8b"))

The root stays at 2 columns however many hours you add, and there is still one linking row. More
hours make each leaf larger (with 4 blocks) or give more leaves (with 8). This is the property
that makes energy models a good fit: they grow along time, and time goes into the leaves.

Try `T=168`, a week, with 7 blocks of one day each.

### Exercise 2: another technology

`sample_data.py` describes a technology as one row: investment cost, variable cost, emissions and
availability profile. `SOLAR` is a row prepared for you; adding it is one line.

In [ ]:
from sample_data import SOLAR, TECHS

with_solar = TECHS | {"solar": SOLAR}
print(with_solar)
workshop.visualize(export(build(techs=with_solar), "pyomo_solar"))

One more root column, `cap[solar]`, and every leaf grows by one `gen` column and one limit row
per hour. The linking row stays alone: the carbon budget now touches solar too, but it was
linking already.

Invent your own technology: a tuple `(investment, variable, emission, profile)` with one of the
profiles `"wind"`, `"solar"` or `"firm"`. A cheap, dirty one such as `(0.2, 0.8, 0.9, "firm")`
changes the answer a lot and the structure not at all.

### Exercise 3: storage

`build(storage=True)` adds the storage from 00. Read the `if storage:` part of the model first.
The constraint to look at is the level:

```python
m.soc[t] == (m.soc[t - 1] if t > 0 else 0) + efficiency * m.charge[t] - m.discharge[t]
```

Inside a block this row is local. In the first hour of a block it uses `soc[t - 1]` from the
previous block. Predict the number of linking rows, then run:

In [ ]:
workshop.visualize(export(build(storage=True), "pyomo_storage"))

Four linking rows: the carbon budget, plus one storage level per block boundary. Those three are
2-link rows: each touches only two neighbouring blocks, and you can see them in the picture as
short strokes at the bottom, right at the corners between two blocks. PIPS-IPM++ handles them
much more cheaply than the carbon budget, which touches every block.

The storage capacity `sto_cap` is the third root column; the model puts it into block `0`
explicitly.

### Exercise 4: an annotation of your own

The Suffix is only one way to give the blocks. `var_block=` takes any mapping from variable to
block, and a `ComponentMap` keeps the annotation outside the model, which is handy when the model
is built by code you do not own.

Here every fourth hour goes to the same block. A perfectly valid split, and a very poor one for a
model with storage:

In [ ]:
from pyomo.common.collections import ComponentMap


def round_robin(model, n_blocks=4):
    blocks = ComponentMap(model.pips_block.items())       # start from the Suffix
    for t in model.T:
        for v in [model.gen[g, t] for g in model.G] + [model.charge[t], model.discharge[t],
                                                       model.soc[t]]:
            blocks[v] = 1 + t % n_blocks
    return blocks


model = build(storage=True)
workshop.visualize(export(model, "pyomo_roundrobin", var_block=round_robin(model)))

Now every storage level links two blocks, and the bottom border is as tall as a leaf. Most of
those rows are still 2-link (block 1 to 2, 2 to 3, 3 to 4). The ones from block 4 back to block 1
are not, because blocks 4 and 1 are not neighbours.

The model is the same and its answer is the same. Only the split changed, and with it how much
work the borders cost.

### Exercise 5: your own change

Some ideas, each with a prediction to check against the picture:

- Make the storage cyclic, so the last hour feeds the first: in the storage level replace
  `else 0` by `else m.soc[T - 1]` in the model file, run both cells of section 1
  again, and export `build(storage=True)`.
- Split into regions or scenarios instead of hours: add an index, then decide the blocks by it.
- Blocks of different lengths: change the rule `1 + t // hours` to something else.

<details><summary>💡 Answer: what does the cyclic storage do?</summary>

It adds one linking row that ties the last hour of block 4 to the first hour of block 1. That row
touches two blocks, but not two neighbouring ones, so it is not 2-link: the summary reports 5
linking rows, of which 3 are 2-link. One such row is harmless. A model with many of them is worth
reformulating.
</details>

## 4. The answer, with HiGHS

Before solving with PIPS-IPM++, get a reference answer from HiGHS, which runs everywhere:

In [ ]:
import pyomo.environ as pyo

reference = {}
for label, model in (("base", build()), ("storage", build(storage=True))):
    pyo.SolverFactory("appsi_highs").solve(model)
    reference[label] = pyo.value(model.obj)
    print(f"{label:8s} objective {reference[label]:.6f}   capacities",
          {g: round(pyo.value(model.cap[g]), 4) for g in model.G})

Storage lowers the cost from about 2.74 to about 2.12: it shifts wind into the hours without
wind, and no gas is built at all.

## 5. Solve with PIPS-IPM++

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped.

In [ ]:
%%requires solver
from pyomo.contrib.solver.common.factory import SolverFactory

model = build()
results = SolverFactory("pipsipmpp").solve(model, solver_options={"SILENT": True})

In [ ]:
%%requires solver
print(results.termination_condition)
print(f"objective  {results.incumbent_objective:.6f}   (HiGHS {reference['base']:.6f})")
print("capacities", {g: round(pyo.value(model.cap[g]), 4) for g in model.G})

The same answer as HiGHS, to about six figures. PIPS-IPM++ is an interior-point method and stops
within a tolerance of the optimum, while HiGHS ends on an exact vertex, so the last digits may
differ.

The solution is loaded onto the model as usual, so `pyo.value(...)` works afterwards; duals and
reduced costs come through the results object's solution loader.

The solve cell shows what PIPS-IPM++ prints: its parameters, a presolve summary and the residuals
at the end. `SILENT` removes most of the iteration log but not those, which is why the results
get a cell of their own.

<details><summary>💡 Hint: <code>convergenceCriteriaSatisfied</code>, not "optimal"?</summary>

That is the name `pyomo.contrib.solver` uses for a successful solve by an interior-point method:
it stopped because the optimality tolerances are met. Treat it as optimal. A run that hits the
iteration limit reports `iterationLimit` and still returns a point.
</details>

<details><summary>💡 Hint: I get <code>'NoneType' object has no attribute 'solve'</code></summary>

That is Pyomo's way of saying the solver is not there: `SolverFactory("pipsipmpp")` returns `None`
instead of raising when the solver cannot be loaded. Run `workshop.check()` to see why.
</details>

### The split changes the work, never the answer

The storage model, solved with the contiguous split, the round robin from exercise 4, and a few
block counts:

In [ ]:
%%requires solver
pips = SolverFactory("pipsipmpp")
splits = {}
for n_blocks in (2, 4, 6, 12):
    model = build(storage=True, n_blocks=n_blocks)
    res = pips.solve(model, solver_options={"SILENT": True})
    splits[f"{n_blocks} contiguous blocks"] = res.incumbent_objective

model = build(storage=True)
res = pips.solve(model, var_block=round_robin(model), solver_options={"SILENT": True})
splits["round robin"] = res.incumbent_objective

In [ ]:
%%requires solver
for label, objective in splits.items():
    print(f"{label:20s} objective {objective:.6f}")
print(f"{'HiGHS':20s} objective {reference['storage']:.6f}")

If changing the split ever changes your objective beyond the solver tolerance, the annotation has
a bug.

### Options

| | |
|---|---|
| `solver_options` | passed to PIPS-IPM++, overriding `options_file` |
| `options_file` | a base configuration, one `NAME value` per line |
| `comm` | the communicator to solve on, by default `MPI.COMM_WORLD` |
| `block_suffix`, `var_block` | where the annotation comes from |
| `n_blocks` | number of leaves, otherwise taken from the annotation |
| `load_solutions` | load the solution onto the model, `True` by default |
| `write_parquet`, `parquet_layout`, `parquet_names` | write the annotated problem while solving |

## 6. More than one process

A notebook is a single process, so everything above solved on one. To use the decomposition, the
same code runs under MPI as a script. One thing is different from the other frameworks:

> Only process 0 builds the model. The others pass `None`.

Building the full model on every process would defeat the point of distributing it, so the
interface expects a model on process 0 and ignores one elsewhere. The solve is still collective:
every process has to reach `solve()`.

In [ ]:
%%writefile pyomo_parallel.py
from mpi4py import MPI
from pyomo.contrib.solver.common.factory import SolverFactory
from pyomo_model import build

comm = MPI.COMM_WORLD
model = build(storage=True) if comm.Get_rank() == 0 else None      # only process 0
results = SolverFactory("pipsipmpp").solve(model, solver_options={"SILENT": True})
if comm.Get_rank() == 0:
    print(f"ranks={comm.Get_size()} {results.termination_condition} "
          f"objective={results.incumbent_objective:.6f}")

In [ ]:
%%requires solver
for ranks in (1, 2, 4):
    workshop.mpirun("pyomo_parallel.py", ranks, show="ranks=")

The same objective on one, two and four processes, within the solver tolerance. The work is
split differently, so the last digits can differ, and so can the number of iterations.

`workshop.mpirun` runs `mpiexec -n 4 python pyomo_parallel.py` for you; in a terminal you can type that yourself.

<details><summary>💡 Hint: how many processes should I use?</summary>

At most one per leaf block, since PIPS-IPM++ refuses more processes than leaves. Fewer is fine;
leaves then share a process. On a laptop, the number of CPU cores is the useful upper limit.
</details>

## 7. Now your own model

Pyomo models rarely carry an obvious block dimension, so the first job is deciding what the
blocks are. Typical choices in energy models:

- time: split the hours, as here. Works when investment is the only thing shared.
- scenario: the classic two-stage stochastic shape, first-stage decisions in the root and one
  leaf per scenario.
- region: one leaf per region, transmission between them as linking rows.

Then annotate, for an indexed model usually a couple of lines:

```python
m.pips_block = pyo.Suffix(direction=pyo.Suffix.LOCAL)
for v in m.component_data_objects(pyo.Var):
    m.pips_block[v] = block_of(v)          # your rule here
```

and export and visualize it with the `export` function from section 2 before solving anything
big. A large root, or leaves of very different sizes, is worth fixing first.

## 8. Checkpoint

- Why are constraints never annotated?
- How many linking rows does the storage create with 8 contiguous blocks, and how many with 8
  round-robin blocks?

## Next

[03 Structure detection](03-structure-detection.ipynb), which finds a structure in a model that
states none.

## Notes

- Linear, continuous models only. PIPS-IPM++ solves LPs and does not branch.
- Maximisation is handled for you: the objective's sign is flipped back on the way out.